# EquivAug: minimal end-to-end example

This notebook runs the whole framework on a handful of examples on a single GPU:

1. **EquivAug data generation** for GSM8K and FOLIO
   (Step 1 Symbolic Representation → Step 2 Equivalence Transformation → Step 3 Natural Language Reconstruction)
2. **SFT** of a small model (`Qwen/Qwen2.5-0.5B`) on a subset of the released EquivAug data
3. **Evaluation** of the SFT model with vLLM
4. **GRPO** with verl for two steps

Requirements: the environment from the README. Optional:

- An LLM endpoint for Steps 1 and 3: `OPENAI_API_KEY` (gpt-4o-mini), or any OpenAI-compatible server via
  `BASE_URL` + `LLM_MODEL`. Without one, the LLM steps load cached gpt-4o-mini outputs from `examples/sample_data/`.
- `prover9` / `mace4` on `PATH` and access to `yale-nlp/FOLIO` for the FOLIO symbolic steps. Without them, the FOLIO
  steps load cached outputs.

The AST transformation of GSM8K (Step 2), SFT, evaluation and GRPO always run.

In [ ]:
import json
import os
import shutil
import subprocess

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "examples" else os.getcwd()
SAMPLE = os.path.join(ROOT, "examples", "sample_data")
OUT = os.path.join(ROOT, "outputs", "demo")
os.makedirs(OUT, exist_ok=True)

# LLM used in Steps 1 and 3
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")
BASE_URL = os.environ.get("BASE_URL")            # e.g. http://localhost:8000/v1
USE_LLM = bool(os.environ.get("OPENAI_API_KEY") or BASE_URL)
HAS_PROVER9 = shutil.which("prover9") is not None and shutil.which("mace4") is not None

SFT_BASE_MODEL = "Qwen/Qwen2.5-0.5B"
LLM_ARGS = f"--model {LLM_MODEL}" + (f" --base_url {BASE_URL}" if BASE_URL else "")


def run(cmd):
    """Run a shell command from the repo root and stream its output."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


def run_or_cache(cmd, cached, target, enabled=True):
    """Run a step; fall back to the cached output if the step cannot run here."""
    if enabled:
        try:
            run(cmd)
            return
        except RuntimeError as e:
            print(f"\n[!] {e}; using cached output instead")
    else:
        print("[!] requirement missing; using cached output")
    shutil.copy(os.path.join(SAMPLE, cached), target)


def load(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


print("repo root  :", ROOT)
print("LLM steps  :", f"{LLM_MODEL} @ {BASE_URL or 'OpenAI'}" if USE_LLM else "cached outputs")
print("Prover9    :", "found" if HAS_PROVER9 else "not found (cached outputs)")

## 1. EquivAug on GSM8K

### Step 1: Symbolic Representation

The LLM rewrites each natural-language solution as one arithmetic expression (an AST), substituting every
intermediate result by the expression that produced it. The expression is evaluated and kept only if it equals the
gold answer.

In [ ]:
G = os.path.join(OUT, "gsm8k")
os.makedirs(G, exist_ok=True)
shutil.copy(os.path.join(SAMPLE, "gsm8k_train.jsonl"), f"{G}/train.jsonl")

run_or_cache(
    f"python step1_symbolic_representation/gsm8k/translate_ast.py "
    f"--input {G}/train.jsonl --output {G}/train_expr.jsonl {LLM_ARGS}",
    cached="gsm8k_step1_expr.jsonl", target=f"{G}/train_expr.jsonl", enabled=USE_LLM)

for r in load(f"{G}/train_expr.jsonl")[:2]:
    print("\nQUESTION:", r["question"])
    print("SOLUTION:", r["answer"])
    print("AST EXPR:", r["expr"])

### Step 2: Equivalence Transformation

The expression is parsed into an AST and rewritten with the equivalence rules of Table 1 (commutativity,
associativity, distributivity, fraction rules, ...). Every variant is evaluated and kept only if its value equals the
original one. This step is purely symbolic.

In [ ]:
run(f"python step2_equivalence_transformation/gsm8k/ast_transformation.py "
    f"--input {G}/train_expr.jsonl --output {G}/train_expr_variants.jsonl "
    f"--max_transformations 1 --max_variants 5")

for r in load(f"{G}/train_expr_variants.jsonl")[:2]:
    print("\nORIGINAL:", r["expr_original"])
    for v in r["expr_variants_with_depth"]:
        print("  variant:", v["expr"], "=", v["value"])

### Step 3: Natural Language Reconstruction

The LLM rewrites the original solution so that its computation order follows each new expression. A solution is kept
only if its final answer is still correct.

In [ ]:
run_or_cache(
    f"python step3_nl_reconstruction/gsm8k/translate_back_gsm8k.py "
    f"--input {G}/train_expr_variants.jsonl --output_dir {G} --per_question_max_expr 5 {LLM_ARGS}",
    cached="gsm8k_step3_explained_depth_1.jsonl", target=f"{G}/train_expr_explained_depth_1.jsonl",
    enabled=USE_LLM)

gsm8k_aug = load(f"{G}/train_expr_explained_depth_1.jsonl")
print(f"{len(gsm8k_aug)} augmented solutions for {len({r['question'] for r in gsm8k_aug})} problems\n")
q = gsm8k_aug[0]["question"]
print("QUESTION:", q)
for r in [r for r in gsm8k_aug if r["question"] == q][:3]:
    print(f"\n--- expression {r['expr']} ---\n{r['expr_explanations'][0]}")

## 2. EquivAug on FOLIO

### Step 1: Symbolic Representation

FOLIO's FOL annotations are converted to Prover9 syntax and the gold label is verified:
True → Prover9 proves the conclusion; False → Prover9 proves its negation; Uncertain → Mace4 finds models for both.

In [ ]:
F = os.path.join(OUT, "folio")
os.makedirs(F, exist_ok=True)

run_or_cache(
    f"python step1_symbolic_representation/folio/fol_to_prover9.py "
    f"--max-examples 10 --timeout 10 --output {F}/folio_prover9_results.jsonl",
    cached="folio_step1_prover9_results.jsonl", target=f"{F}/folio_prover9_results.jsonl",
    enabled=HAS_PROVER9)

for r in load(f"{F}/folio_prover9_results.jsonl")[:10]:
    print(f"example {r['example_id']:>5}: gold={r['label']:<10} solver={r['predicted_label']}")

### Step 2: Equivalence Transformation

The premises are rewritten with the logical equivalences of Table 2 (premise reordering, contraposition,
De Morgan's laws, implication expansion, ...) and combined with different Prover9 search strategies. Each transformed
problem is re-proved, so every proof path is valid but takes a different route.

In [ ]:
run_or_cache(
    f"python step2_equivalence_transformation/folio/folio_transform_prover9.py "
    f"--input {F}/folio_prover9_results.jsonl --output {F}/folio_diverse_proofs_p9.jsonl "
    f"--timeout 10 --num-strategies 2 --num-transforms 4",
    cached="folio_step2_diverse_proofs.jsonl", target=f"{F}/folio_diverse_proofs_p9.jsonl",
    enabled=HAS_PROVER9)
run_or_cache(
    f"python step2_equivalence_transformation/folio/merge_to_sft.py "
    f"--diverse {F}/folio_diverse_proofs_p9.jsonl --original {F}/folio_prover9_results.jsonl "
    f"--output {F}/folio_sft_data.jsonl",
    cached="folio_step2_sft_data.jsonl", target=f"{F}/folio_sft_data.jsonl")

ex = load(f"{F}/folio_diverse_proofs_p9.jsonl")[0]
print(f"example {ex['example_id']} ({ex['label']}): {ex['num_proofs']} proof paths")
for p in ex["proofs"][:2]:
    print(f"\n--- strategy={p.get('strategy')}  transform={p.get('transform')} ---")
    print(p.get("proof", "")[:800])

### Step 3: Natural Language Reconstruction

Each proof trace is translated into a natural-language explanation; it is kept only if its verdict matches the gold
label. Uncertain problems have no proof trace, so the `balance` mode adds explanations for them.

In [ ]:
def folio_step3():
    run(f"python step3_nl_reconstruction/folio/translate_back_folio.py translate "
        f"--input {F}/folio_sft_data.jsonl --output {F}/folio_sft_nl.jsonl --test-limit 8 {LLM_ARGS}")
    run(f"python step3_nl_reconstruction/folio/translate_back_folio.py balance "
        f"--input {F}/folio_sft_nl.jsonl --output {F}/folio_sft_nl_balanced.jsonl --target-ratio 0.7 {LLM_ARGS}")

if USE_LLM:
    try:
        folio_step3()
    except RuntimeError as e:
        print(f"[!] {e}; using cached output instead")
        shutil.copy(os.path.join(SAMPLE, "folio_step3_sft_nl.jsonl"), f"{F}/folio_sft_nl_balanced.jsonl")
else:
    shutil.copy(os.path.join(SAMPLE, "folio_step3_sft_nl.jsonl"), f"{F}/folio_sft_nl_balanced.jsonl")

folio_aug = load(f"{F}/folio_sft_nl_balanced.jsonl")
r = folio_aug[0]
print("PREMISES  :", r["premises_nl"])
print("CONCLUSION:", r["conclusion_nl"], f"[{r['label']}]")
print("\nEXPLANATION:\n" + r["explanation"])

## 3. Supervised fine-tuning

We fine-tune `Qwen/Qwen2.5-0.5B` for one epoch on 256 examples of the released GSM8K EquivAug data
(`data/gsm8k/equivaug.jsonl`). The full runs use `sft/run_sft.sh` (see README).

In [ ]:
SFT_DATA = f"{OUT}/sft_subset.jsonl"
with open(os.path.join(ROOT, "data/gsm8k/equivaug.jsonl")) as fin, open(SFT_DATA, "w") as fout:
    for i, line in enumerate(fin):
        if i >= 256:
            break
        fout.write(line)

SFT_MODEL = f"{OUT}/sft_model"
run(f"python sft/trained_sft_gsm8k.py --model_name_or_path {SFT_BASE_MODEL} --data_path {SFT_DATA} "
    f"--output_dir {SFT_MODEL} --num_train_epochs 1 --per_device_train_batch_size 8 "
    f"--gradient_accumulation_steps 1 --learning_rate 1e-5 --warmup_steps 2 --max_seq_length 1024 "
    f"--validation_split 0 --save_strategy no --logging_steps 8 --bf16")

## 4. Evaluation

Sample 2 responses for 20 GSM8K test questions with vLLM and compute the accuracy
(the paper uses 5 responses on the full test set: `evaluation/run_eval.sh`).

In [ ]:
EVAL_DIR = f"{OUT}/eval"
run(f"python evaluation/gsm8k/generate_vllm.py --model-path {SFT_MODEL} --max-samples 20 "
    f"--num-responses 2 --max-new-tokens 512 --gpu-mem-util 0.6 --output-dir {EVAL_DIR}")
run(f"python evaluation/gsm8k/compute_accuracy.py "
    f"--results-path {EVAL_DIR}/gsm8k_test_2responses_512tokens.pkl")

## 5. GRPO

Two GRPO steps with verl on GSM8K, starting from the SFT model. `grpo/run_grpo_gsm8k.sh` builds the parquet files
and launches `verl.trainer.main_ppo`; the extra arguments shrink the run.

In [ ]:
run(f"N_GPUS=1 RAY_NUM_CPUS=4 MICRO_BS=4 GPU_MEM_UTIL=0.4 LOGGER='[\"console\"]' DATA_DIR={OUT}/grpo_data "
    f"bash grpo/run_grpo_gsm8k.sh {SFT_MODEL} demo "
    f"data.train_batch_size=16 actor_rollout_ref.actor.ppo_mini_batch_size=16 "
    f"actor_rollout_ref.rollout.n=4 data.max_response_length=512 "
    f"trainer.total_training_steps=2 trainer.val_before_train=False trainer.test_freq=-1 trainer.save_freq=-1 "
    f"trainer.default_local_dir={OUT}/grpo_checkpoints")